In [8]:
import numpy as np
import plotly.graph_objects as go

# Link lengths (mm)
a1, a2, a3 = 37.0, 100, 300.0
Dmax = a1 + a2 + a3
Dmin = abs(a1 - (a2 + a3))

# Angular samples
theta = np.linspace(np.deg2rad(-46), np.deg2rad(46), 300)

# Unit circle samples for boundaries
unit_outer = np.vstack((np.cos(theta), np.sin(theta)))
unit_inner = np.vstack((np.cos(theta[::-1]), np.sin(theta[::-1])))

# z0 values for slider
z_values = np.linspace(0, Dmax, 101)

# Create frames for each z0 with hover information
frames = []
for z0 in z_values:
    if z0 > Dmax:
        xs, ys = [], []
    else:
        r_max = np.sqrt(Dmax**2 - z0**2)
        r_min = np.sqrt(Dmin**2 - z0**2) if z0 < Dmin else 0.0
        outer = unit_outer * r_max
        if r_min > 0:
            inner = unit_inner * r_min
            xs = np.concatenate((outer[0], inner[0]))
            ys = np.concatenate((outer[1], inner[1]))
        else:
            xs, ys = outer

    trace = go.Scatter(
        x=xs, y=ys, fill='toself',
        hovertemplate=(
            "x: %{x:.1f} mm<br>"
            "y: %{y:.1f} mm<br>"
            f"z: {z0:.1f} mm<extra></extra>"
        ),
        showlegend=False
    )
    frames.append(go.Frame(data=[trace], name=f"{z0:.1f}"))

# Initial plot at z0 = 0
init_z = z_values[0]
init_r_max = np.sqrt(Dmax**2 - init_z**2)
init_outer = unit_outer * init_r_max
init_trace = go.Scatter(
    x=init_outer[0], y=init_outer[1], fill='toself',
    hovertemplate=(
        "x: %{x:.1f} mm<br>"
        "y: %{y:.1f} mm<br>"
        f"z: {init_z:.1f} mm<extra></extra>"
    ),
    showlegend=False
)

fig = go.Figure(data=[init_trace], frames=frames)

# Slider steps
steps = [
    {
        "label": f"{z0:.1f}",
        "method": "animate",
        "args": [
            [f"{z0:.1f}"],
            {"mode": "immediate", "frame": {"duration": 0}, "transition": {"duration": 0}}
        ],
    }
    for z0 in z_values
]

sliders = [
    {
        "active": 0,
        "currentvalue": {"prefix": "z0 (mm): "},
        "pad": {"t": 50},
        "steps": steps
    }
]

# Add Play button, slider, and set square large size
fig.update_layout(
    title="Leg Workspace vs. z0",
    width=900,
    height=900,
    xaxis_title="X (mm)",
    yaxis_title="Y (mm)",
    xaxis=dict(scaleanchor="y", scaleratio=1),
    updatemenus=[
        {
            "type": "buttons",
            "buttons": [
                {
                    "label": "Play",
                    "method": "animate",
                    "args": [None, {"frame": {"duration": 100, "redraw": True}, "fromcurrent": True}],
                }
            ],
            "pad": {"r": 10, "t": 10},
            "showactive": False,
            "x": 0.0,
            "y": -0.1
        }
    ],
    sliders=sliders,
    margin=dict(l=50, r=50, t=50, b=50)
)

fig.show()
